In [2]:
import pdfplumber
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from dataclasses import dataclass, field
import re
import logging

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
@dataclass
class ParsedDocument:
    domain: str
    source_file: str
    sections: list[dict] = field(default_factory=list)
    # Each section: {"id": "591-3", "title": "...", "text": "...", "parent": "Part II"}

In [5]:
def clean_text(text: str) -> str:
    """Normalize whitespace and remove common PDF artifacts."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)  # fix hyphenation
    text = text.strip()
    return text

def _open_section(match, current_article, page_num, doc) -> dict:
    """Save current section (if any) and open a new one."""
    title = match.group(2).strip()
    return {
        "id": match.group(1),
        "title": title,
        "text": "",
        "parent": current_article,
        "page": page_num + 1,
        "_title_complete": _title_looks_complete(title),
    }


def _title_looks_complete(title: str) -> bool:
    """
    A heading title is complete if it ends with sentence-ending punctuation
    or a closing bracket. Incomplete titles get continued on the next line.
    """
    return bool(re.search(r'[.!?\]]$', title.strip()))

In [6]:
# ── Line classifiers ───────────────────────────────────────────────────────────

HEADER_PATTERNS = [
    re.compile(r'^TORONTO MUNICIPAL CODE$'),
    re.compile(r'^CHAPTER \d+,'),
]

# Matches: "547-2 January 1, 2025" or "591-4 March 15, 2024"
FOOTER_PATTERN = re.compile(r'^\d{3}-\d+\s+\w+ \d+, \d{4}$')

# Matches: "ARTICLE 1", "ARTICLE 2", "ARTICLE V" etc.
ARTICLE_PATTERN = re.compile(r'^ARTICLE\s+(\d+|[IVXLC]+)$', re.IGNORECASE)

# Matches real section headings: "§ 547-1.1. Title text"
# The § must be at the start of the line, and title must start with capital letter
SECTION_HEADING = re.compile(r'^§\s+(\d{3}-\d+(?:\.\d+)?)\.\s*([A-Z].*)')


def is_header(line: str) -> bool:
    return any(p.match(line) for p in HEADER_PATTERNS)


def is_footer(line: str) -> bool:
    return bool(FOOTER_PATTERN.match(line))


def is_toc_line(line: str) -> bool:
    """
    ToC lines look identical to real headings — they start with §.
    The difference is that in the ToC, § lines are never followed by prose.
    We handle this at the document level, not the line level.
    """
    return bool(SECTION_HEADING.match(line))

In [7]:
domain = 'Short Term Rental'
# pdf_path = '../data/raw/ch547_short_term_rental.pdf'
pdf_path = "./data/raw/ch591_noise.pdf"

In [8]:
doc = ParsedDocument(domain=domain, source_file=pdf_path)

current_article = None
current_section = None

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        raw_text = page.extract_text()
        if not raw_text:
            continue

        for line in raw_text.split('\n'):
            line = clean_text(line)
            if not line or is_header(line) or is_footer(line):
                continue

            article_match = ARTICLE_PATTERN.match(line)
            section_match = SECTION_HEADING.match(line)

            if article_match:
                current_article = f"ARTICLE {article_match.group(1).upper()}"
                continue

            if section_match:
                # Save completed section
                if current_section:
                    doc.sections.append(current_section)

                current_section = {
                    "id": section_match.group(1),
                    "title": section_match.group(2).strip(),
                    "text": "",
                    "parent": current_article,
                    "page": page_num + 1,
                }
                continue

            # Body text — accumulate into current section
            if current_section is not None:
                current_section["text"] += " " + line

# Don't lose the last section
if current_section:
    doc.sections.append(current_section)

logger.info(f"  → {len(doc.sections)} sections across "
            f"{len(set(s['parent'] for s in doc.sections))} articles ({domain})")

INFO:__main__:  → 39 sections across 5 articles (Short Term Rental)


In [9]:
# Write doc to local file for inspection
output_path = Path(f"./data/debug/{Path(pdf_path).stem}_parsed.json")
output_path.parent.mkdir(exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    import json
    json.dump(doc.__dict__, f, indent=2, ensure_ascii=False)